`torch.autograd` 是 PyTorch 的自动微分引擎，为神经网络训练提供动力。

## 一.简单背景介绍
更多详细内容, 请参考markdown以及其他网络视频

神经网络（NN）是作用于输入数据的一系列嵌套函数集合。这些函数由*参数*（包含权重和偏置）定义，在 PyTorch 中这些参数存储在张量（tensors）中。<br>
神经网络的训练分为两步<br>
**前向传播（Forward Propagation）**：在前向传播中，神经网络对正确的输出做出最佳猜测。它将输入数据传入每一层函数以得出此猜测。<br>
**反向传播（Backward Propagation）**：在反向传播中，神经网络根据其猜测的误差比例调整参数。它通过从输出端向后遍历，收集误差相对于函数参数的导数（*梯度*），并使用梯度下降优化参数。<br>

## 二.PyTorch 中的使用方法

让我们看一个单一的训练步骤。在此示例中，我们从 `torchvision` 加载一个预训练的 resnet18 模型。<br>
我们创建一个随机数据张量来代表单个图像（3 通道，高宽均为 64），并将其对应的 `label` 初始化为随机值。预训练模型中的标签形状为 (1,1000)。<br>

> Attention:<br>
> 此教程*在较老版本的Pytorch上(1.x版本)*<br>
> 仅在 CPU 上运行，不适用于 GPU 设备（即使将张量移动到 CUDA 设备上也不行）。<br>

### 1.导入模型, 输入材料, 及对应标签
模型初始参数为预训练好的, 并非全随机化

In [9]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1, 3, 64, 64)
labels = torch.rand(1, 1000)

#### <1>. ResNet18 是什么?

> 一个 CNN 架构,2015 年微软的 **何恺明(Kaiming He)** 团队提出的.<br>
> 它解决了一个当时困扰业界多年的问题:**网络越深反而越难训练**。深层网络梯度一路反传会越来越小 ,训练不动。<br>
> ResNet 的答案是一个绝妙的 trick——**残差连接(跳跃连接)**:<br>

```text
普通层块:   输出 = F(x)           ← 直接学"完整映射"
ResNet块:   输出 = F(x) + x       ← 只学"差值",再把原来的 x 加回来
```

> 这个 `+ x` 就是捷径(skip connection)。梯度可以**绕过中间的层,直接顺着捷径流回去**,深层的梯度消失问题被物理性化解。<br>
> ResNet 一战封神,直到今天几乎所有现代模型(包括后面你会接触的迁移学习)的骨架都是它的变体。<br>

> **"18"** = 18 个带权重的层(卷积+全连接)。家族里有 18 / 34 / 50 / 101 / 152,数字越大越深越强但越贵。<br>
> 实测显示它约 **1170 万**个参数——不大不小,正好适合入门。<br>

#### <2>. `weights=ResNet18_Weights.DEFAULT` 是在初始化 W 吗?

**准确来说:不是"随机初始化",而是"加载预训练权重"。**

> 已经学过的 Xavier/MSRA 是**随机初始化**——网络第一次见到数据前的随机 W,适合从零训练。<br>
> 而 `ResNet18_Weights.DEFAULT` 加载的是官方在 **ImageNet 数据集**(128 万张图、1000 个类别)上**训练完成的现成权重**。<br>

> 换句话说,它一出生就不是"白纸",而是"已经见过世界、懂点视觉的熟手"。这整个机制叫 **迁移学习(Transfer Learning)**:<br>
> - 随机初始化:像婴儿从头学认物,要喂海量数据<br>
> - 预训练权重:像请了个已经会认 1000 种东西的老师傅,你只需要让它"学你的一门小生意"(比如猫狗)<br>

**这正是未来做猫狗识别的捷径。** 到时, 我们不从零训练一个 CNN,而是加载 ResNet18 的预训练权重,把最后那层 1000 类输出换成 2 类,然后微调——几天的工作量缩到几小时,准确率还更高。



#### <3>.关于`data`和`labels`的形状与维度
- `data = torch.rand(1, 3, 64, 64)` 是四维吗?
  > **是,而且是深度学习最标准的"图片批次"格式: `(N, C, H, W)`**<br>
  > | 维度  | 含义                     | 本例的值      |
  > | ----- | ------------------------ | ------------- |
  > | **N** | batch size(一次喂几张图) | 1(就 1 张)    |
  > | **C** | 通道数(几层颜色/特征)    | 3(RGB 三通道) |
  > | **H** | 高(像素)                 | 64            |
  > | **W** | 宽(像素)                 | 64            |
<br>

- `labels = torch.rand(1, 1000)` 是二维吗?
  > 是,形状是 (batch=1, 类别数=1000)。<br>
  > 因为 ImageNet 恰好有 1000 个类别,模型的输出 `out` 也是 `(1, 1000)`——**每一格是该图属于第 i 类的分数(logit)**。<br>
  > 这个 `labels` 就是个随机生成的"假答案",让教程能演示 `loss = loss_fn(out, labels)` 怎么算。<br>
  
  对应到未来的猫狗项目,输出和标签都会变成 `(N, 2)`——一个格是"猫"的分数,一个格是"狗"的分数。

### 2.(手动进行一次)预测
接下来，我们将输入数据传入模型，使其通过每一层以进行预测。这就是前向传播（forward pass）。

In [2]:
prediction = model(data) # 前向传播

### 3.(手动进行一次)求导

- 我们使用模型的预测值和对应的标签来计算误差（loss）。<br>
  > 即定义优化目标函数:$J(W) = \underbrace{\text{Loss}}_{\text{数据项}} + \underbrace{\lambda R(W)}_{\text{正则项}}$<br>
  > 而下文中`loss = (prediction - labels).sum()`<br>
  > 直接默认了 $J(W) = Loss$ 就是 `(prediction - labels).sum()` 这个标量<br>
- 下一步是将该误差通过网络进行反向传播。
  > 当我们对误差张量调用 `.backward()` 时，反向传播正式启动。<br>
  > 此时 Autograd 会计算每个模型参数的梯度，并将其存储在参数的 `.grad` 属性中。<br>

In [3]:
loss = (prediction - labels).sum()
loss.backward() # backward pass

- tips:这个 demo 教程只作演示使用, 不要被其定义的$Loss$带偏<br>

    **`(prediction - labels).sum()` 不是一个正经的损失函数。** 这只是作者拿来演示随便弄的Loss函数:<br>

    ```python
    loss = (prediction - labels).sum()
    ```

    > 举一个简单例子:<br>
    > 假设`prediction` = [5, -3],而`labels` = [1, 1]<br>
    > 按照这个Loss函数, 则`loss` = [4, -4],求和是 0<br>
    > **正负误差相互抵消,loss 变 0 但预测完全不对**。<br>
    
    这在真实任务里是荒谬的。

    所以这个 demo 里它扮演的角色是:**"随便一个可导的标量,拿来演示 `.backward()` 机制"**。<br>
    它不教你"损失函数应该长什么",而是教你"autograd 会对着任何标量反传梯度"。<br>

### 4.(手动进行一次)梯度下降计算
接下来，我们加载一个优化器，本例中使用学习率为 0.01 且[动量（momentum）](https://medium.com/data-science/stochastic-gradient-descent-with-momentum-a84097641a5d)为 0.9 的 SGD。<br>
我们将模型的所有参数注册到优化器中。<br>

In [4]:
optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

- `model.parameters()`:模型中所有需要被优化的对象<br>
  模型里, 在这一步所有** "需要被学习的参数" **的集合。<br>
  > 例如, 对一个 `nn.Linear(4, 2)`,可学习参数有两个:
  > | 名字     | 形状     | 含义       |
  > | -------- | -------- | ---------- |
  > | `weight` | `(2, 4)` | 权重矩阵 W |
  > | `bias`   | `(2,)`   | 偏置 b     |

  有些参数的学习并不是在这一步进行的, 比如BN的`running_mean`这种不需要梯度的就不在名单里.<br>
  这份名单只会收录所有属性为`requires_grad=True`的参数.<br>

  对ResNet18, 这份名单就是 **1170 万个参数**的超级大名单(weight 各层都有,加上卷积的,还有 BN 层的 $\gamma 和 \beta$,全算在内)

- `lr=1e-2`:步长(学习率)<br>
  `lr`, 即*learning rate*的缩写:<br>

  - **太小**(1e-4)→ 磨蹭半天不收敛,训练慢
  - **太大**(1.0)→ 步子跨过山底,冲上对面山坡,来回震荡甚至发散(loss 越训越大)
  - `1e-2 = 0.01`,是个常见的安全起步值

### 5.(手动进行一次)执行参数更新
最后，我们调用 `.step()` 来启动梯度下降。优化器会根据存储在 `.grad` 中的梯度来调整每个参数。<br>

In [5]:
optim.step() #gradient descent(梯度下降)

> Warning:<br>
>
> 真实训练是循环:每轮都要 `optimizer.zero_grad()` 清零 → 前向算 loss → `loss.backward()` 累积梯度 → `optimizer.step()` 更新。<br>
> 教程只走一步,所以省了清零,但自己写训练循环时**绝不能省**。<br>

## 三.Autograd 中的微分

让我们看看 `autograd` 是如何收集梯度的。<br>
我们创建两个 `requires_grad=True` 的张量 `a` 和 `b`。这向 `autograd` 发出信号，表明应对它们执行的所有操作进行跟踪。<br>

In [2]:
import torch

a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)

我们通过 `a` 和 `b` 创建另一个张量 `Q`。
$$
Q=3a^3-b^2
$$

In [3]:
Q = 3*a**3 - b**2

假设 `a` 和 `b` 是神经网络的参数，而 `Q` 是误差。在神经网络训练中，我们需要误差相对于参数的梯度(偏微分)，即：
$$
\frac{\partial Q}{\partial a} = 9a^2
$$
$$
\frac{\partial Q}{\partial b} = -2b
$$
当我们对 `Q` 调用 `.backward()` 时，autograd 会计算这些梯度并将其存储在各自张量的 `.grad` 属性中。

由于 `Q` 是一个向量，我们需要在 `Q.backward()` 中显式传递一个 `gradient` 参数。`gradient` 是一个与 `Q` 形状相同的张量，它代表 Q 相对于其自身的梯度(全微分)，即：
$$
\frac{\mathrm{d}Q}{\mathrm{d}Q} = 1
$$
同样地，我们也可以将 Q 聚合为一个标量，然后隐式调用 backward，例如 `Q.sum().backward()`。

In [4]:
external_grad = torch.tensor([1., 1.])
Q.backward(gradient=external_grad)

梯度现在已存入 `a.grad` 和 `b.grad` 中。

In [5]:
# 检查收集到的梯度是否正确
print(9*a**2 == a.grad)
print(-2*b == b.grad)

tensor([True, True])
tensor([True, True])


- Q: 这里为什么`.backward()`要传个张量?

  - 教程里 Q 其实是**扮演 Loss 的角色**(原文写"假设 a 和 b 是神经网络的参数,而 Q 是误差"). <br>
    只不过真实 Loss 一定会收敛成标量(比如对整个 batch 求平均),而这里 Q **偷懒保留了向量形态**,没有聚合。<br>
    所以准确说:**Q 是"一个没做标量化的 Loss"**。<br>
 
  - 不管把 Q 看成"没标量化的 Loss"还是网络的某个中间输出,**本质都一样:它是向量,`backward` 缺一个"初始权重"(上游梯度)就不知道从哪起步。** <br>

- Q: 为什么`.backward()`传`[1,1]`呢?

  - 数学上, 当Q是个向量时, 要对其下层的向量也做求导, 算的是 **"Σ gradientᵢ·Qᵢ"这个标量**对 a、b 的偏导。传 `[1,1]`,就是求:<br>

    ```
    S = 1·Q₁ + 1·Q₂ = Q₁ + Q₂     ← 等价于 Q.sum()
    ∂S/∂a = [36, 81],  ∂S/∂b = [-12, -8]
    ```
    <br>
    传 `[2,3]` 就是求 `2·Q₁ + 3·Q₂` 的偏导。<br>



In [18]:
# demo A: 聚合成标量再 backward, 结果和传 [1,1] 一模一样
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)
Q = 3*a**3 - b**2

S = Q.sum()            # S = -12 + 65 = 53, 一个标量
S.backward()           # 隐式 backward, 不传参
print("a.grad =", a.grad.tolist())   # [36.0, 81.0]
print("b.grad =", b.grad.tolist())   # [-12.0, -8.0]

# demo B: 换加权 [2,3], 相当于求 2*Q1 + 3*Q2 的偏导
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)
Q = 3*a**3 - b**2

Q.backward(gradient=torch.tensor([2., 3.]))
print("a.grad =", a.grad.tolist())   # [72.0, 243.0]  = [2*36, 3*81]
print("b.grad =", b.grad.tolist())   # [-24.0, -24.0] = [2*(-12), 3*(-8)]

a.grad = [36.0, 81.0]
b.grad = [-12.0, -8.0]
a.grad = [72.0, 243.0]
b.grad = [-24.0, -24.0]


## 四.计算图（Computational Graph）

从概念上讲，autograd 在一个有向无环图（DAG）中记录了数据（张量）及所有已执行的操作（以及由此产生的新张量），该图由 [Function](https://pytorch.ac.cn/docs/stable/autograd.html#torch.autograd.Function) 对象组成。在这个 DAG 中，叶子节点是输入张量，根节点是输出张量。通过从根节点回溯到叶子节点，你可以利用链式法则自动计算梯度。

在前向传播中，autograd 同时执行两项操作：

- 运行请求的操作以计算结果张量，以及
- 在 DAG 中维护该操作的*梯度函数*。

当对 DAG 根节点调用 `.backward()` 时，反向传播开始。此时 `autograd` 会：

- 根据每个 `.grad_fn` 计算梯度，
- 将它们累加到对应张量的 `.grad` 属性中，并且
- 利用链式法则，一直传播到叶子张量。

下方是我们示例中 DAG 的可视化表示。图中箭头方向为前向传播的方向。节点代表前向传播中每个操作的逆向函数。蓝色的叶子节点代表我们的叶子张量 `a` 和 `b`。<br>
(图被我吃了)

> Attention:<br>
> 
> **PyTorch 中的 DAG 是动态的**：需要注意的一个重要点是，图是在每次调用 `.backward()` 后从头开始重建的。<br>
> 这正是允许你在模型中使用控制流语句的原因；如果需要，你可以在每次迭代中改变形状、大小和操作。<br>

## 五.从 DAG 中排除

`torch.autograd` 会跟踪所有 `requires_grad` 标志设置为 `True` 的张量上的操作。对于不需要梯度的张量，将此属性设置为 `False` 可将其排除在梯度计算 DAG 之外。

即使只有一个输入张量的 `requires_grad=True`，操作的输出张量也需要梯度。

In [14]:
x = torch.rand(5, 5)
y = torch.rand(5, 5)
z = torch.rand((5, 5), requires_grad=True)

a = x + y
print(f"Does `a` require gradients?: {a.requires_grad}")
b = x + z
print(f"Does `b` require gradients?: {b.requires_grad}")

Does `a` require gradients?: False
Does `b` require gradients?: True


在神经网络中，不计算梯度的参数通常被称为冻结参数。如果你预先知道不需要某些参数的梯度，那么“冻结”模型的一部分非常有用（这可以通过减少 autograd 计算提供一些性能优势）。

在微调（finetuning）中，我们会冻结大部分模型，通常只修改分类器层以对新标签进行预测。让我们通过一个小例子来演示。和之前一样，我们加载预训练的 resnet18 模型，并冻结所有参数。

In [15]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
from torch import nn, optim

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze all the parameters in the network(冻结网络中的全部参数)
for param in model.parameters():
    param.requires_grad = False

假设我们想在包含 10 个标签的新数据集上微调模型。在 resnet 中，分类器是最后一层线性层 `model.fc`。我们可以简单地将其替换为一个新的线性层（默认未冻结），作为我们的分类器。

In [16]:
model.fc = nn.Linear(512, 10)

现在模型中的所有参数（除 `model.fc` 的参数外）都被冻结了。唯一计算梯度的参数是 `model.fc` 的权重和偏置。

In [17]:
# Optimize only the classifier(只优化分类器)
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

请注意，尽管我们将所有参数注册到优化器中，但唯一计算梯度（并在梯度下降中更新）的参数是分类器的权重和偏置。

同样的排除功能可以通过上下文管理器 [torch.no_grad()](https://pytorch.ac.cn/docs/stable/generated/torch.no_grad.html) 使用。